# MP4 から before / after 候補を選ぶ

この notebook は **動画探索と候補固定だけ**を行います。Lab解析は `video_selected_pair_lab.ipynb` に分離しました。

`RUN_SEARCH=False` なら既存の探索結果を再利用し、動画探索は実行しません。`RUN_SEARCH=True` のときだけ新規探索を実行します。


In [6]:
from pathlib import Path
import json
import os

cwd = Path.cwd().resolve()
marker = Path('analysis/run_video_roi_search.py')
if (cwd / marker).is_file():
    REPO_ROOT = cwd
elif cwd.name == 'notebooks' and (cwd.parent / marker).is_file():
    REPO_ROOT = cwd.parent
    os.chdir(REPO_ROOT)
else:
    raise RuntimeError(f'ikiikimake のリポジトリ直下または notebooks/ から実行してください: current={cwd}')

print('repo root:', Path.cwd())

repo root: C:\Users\mail\work\ikiikimake


## 設定

`OUTPUT` は動画ファイル名から自動で決めます。`makeup.mp4` なら `outputs/makeup/` です。
探索条件を変えてやり直すときは、既存結果を勝手に上書きせず停止します。必要ならその動画の出力フォルダを明示的に削除してから再実行します。


In [7]:
# ---- 実験条件 ----
VIDEO = Path('makeup0923.mp4')

# メイク前として意味のある時間帯。
BEFORE_RANGE = (207.0, 267.0)

# after は明示範囲か「動画のラストN秒」のどちらか一方。
AFTER_RANGE = (1380.0, 1500.0)
AFTER_LAST_SECONDS = None

INTERVAL = 1.0
REFINE_INTERVAL = 0.5
TOP = 10

# 新規探索なら True。探索後に同じ条件を再利用するときだけ False。
RUN_SEARCH = True

# 目視で採用する順位。まだ選ばないなら None。
APPROVED_RANK = 3


## 探索または既存結果の読み込み


In [8]:
from collections import Counter
from analysis.run_video_roi_search import parse_args, probe_video, resolve_after_range, run

if AFTER_RANGE is not None and AFTER_LAST_SECONDS is not None:
    raise ValueError('AFTER_RANGE と AFTER_LAST_SECONDS は同時に指定できません。')

video_info = probe_video(VIDEO.resolve())
duration = video_info['duration_seconds']
split = duration / 2
resolved_after_range = resolve_after_range(AFTER_RANGE, AFTER_LAST_SECONDS, split, duration)

OUTPUT = Path('outputs') / VIDEO.stem
matching_path = OUTPUT / 'matching.json'
manifest_path = OUTPUT / 'scan_manifest.json'

print(f'video duration: {duration:.3f}s')
print(f'after range   : {resolved_after_range[0]:.3f}〜{resolved_after_range[1]:.3f}s')
print(f'output        : {OUTPUT}')

if RUN_SEARCH:
    if OUTPUT.exists() and any(OUTPUT.iterdir()):
        raise ValueError(
            f'この動画の探索結果が既にあります: {OUTPUT}. '
            '探索条件を変えてやり直す場合は、このフォルダを明示的に削除してから再実行してください。'
        )
    argv = [
        '--video', str(VIDEO),
        '--output', str(OUTPUT),
        '--before-range', str(BEFORE_RANGE[0]), str(BEFORE_RANGE[1]),
        '--interval', str(INTERVAL),
        '--refine-interval', str(REFINE_INTERVAL),
        '--top', str(TOP),
    ]
    if AFTER_LAST_SECONDS is not None:
        argv += ['--after-last-seconds', str(AFTER_LAST_SECONDS)]
    else:
        argv += ['--after-range', str(AFTER_RANGE[0]), str(AFTER_RANGE[1])]
    manifest = run(parse_args(argv))
else:
    if not matching_path.is_file() or not manifest_path.is_file():
        raise FileNotFoundError(f'既存探索結果がありません: {OUTPUT}')
    manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
    expected_ranges = {'before': list(BEFORE_RANGE), 'after': list(resolved_after_range)}
    if manifest.get('selected_ranges') != expected_ranges:
        raise RuntimeError(
            '現在の探索条件と保存済み結果の範囲が一致しません。'
            f' current={expected_ranges} saved={manifest.get("selected_ranges")}'
        )

if not matching_path.is_file() or not manifest_path.is_file():
    raise FileNotFoundError('探索結果ファイルが生成されていません。')

matching = json.loads(matching_path.read_text(encoding='utf-8'))
scan_manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
pairs = matching.get('ranked_pairs', [])
if not pairs:
    reasons = Counter(
        error
        for record in scan_manifest.get('records', [])
        for error in record.get('report', {}).get('errors', [])
    )
    print('比較候補は0件です。自動チェックで落ちた主な理由:')
    for reason, count in reasons.most_common(12):
        print(f'{count:3d}  {reason}')
    raise RuntimeError('指定した時間帯では eligible pair がありませんでした。')

print(f'{len(pairs)} candidates loaded')

video duration: 1728.352s
after range   : 1380.000〜1500.000s
output        : outputs\makeup0923


ValueError: この動画の探索結果が既にあります: outputs\makeup0923. 探索条件を変えてやり直す場合は、このフォルダを明示的に削除してから再実行してください。

## 候補の数値を確認

スコアは幾何差です。小さいほど撮影条件が近い候補で、美しさやメイク効果のスコアではありません。


In [9]:
from IPython.display import HTML, display

columns = [
    ('rank', 'rank'), ('before_time', 'before_time'), ('after_time', 'after_time'),
    ('score', 'score'), ('yaw_gap', 'yaw_gap_degrees'), ('pitch_gap', 'pitch_gap_degrees'),
    ('roll_gap', 'roll_gap_degrees'), ('face_scale_ratio', 'face_scale_ratio'),
    ('roi_rms', 'roi_procrustes_rms'), ('eye_gap', 'eye_aperture_gap'), ('mouth_gap', 'mouth_opening_gap'),
]
headers = ''.join(f'<th>{label}</th>' for label, _ in columns)
rows = []
for rank, pair in enumerate(pairs, 1):
    values = {'rank': rank, 'before_time': pair['before_time'], 'after_time': pair['after_time'], 'score': pair['score'], **pair['terms']}
    cells = ''.join(
        f'<td>{values[key]:.4f}</td>' if isinstance(values[key], float) else f'<td>{values[key]}</td>'
        for _, key in columns
    )
    rows.append(f'<tr>{cells}</tr>')
display(HTML(f'<table><thead><tr>{headers}</tr></thead><tbody>{"".join(rows)}</tbody></table>'))

report = OUTPUT / 'report.html'
if not report.is_file():
    raise FileNotFoundError(report)
print('候補画像は report.html で確認:', report.resolve())

rank,before_time,after_time,score,yaw_gap,pitch_gap,roll_gap,face_scale_ratio,roi_rms,eye_gap,mouth_gap
1,240.9907,1403.4854,0.9303,1.4934,0.3701,2.0631,1.0031,0.0177,0.0049,0.0072
2,223.0145,1403.4854,0.9495,1.4704,0.2628,1.6297,1.0003,0.0162,0.0095,0.0062
3,216.0075,1443.9842,1.0896,2.6396,0.6741,0.8606,1.1134,0.0098,0.0085,0.0010
4,237.9878,1443.9842,1.2061,0.4647,0.3350,0.5737,1.1379,0.0183,0.0025,0.0264
5,213.0045,1478.0182,1.2912,2.4468,0.4071,1.8833,1.1635,0.0153,0.0065,0.0006
6,229.9797,1478.0182,1.3631,0.3351,1.9862,2.9299,1.1686,0.0120,0.0092,0.0026
7,238.9887,1421.0029,2.8392,4.1424,8.4953,4.1928,1.0102,0.0459,0.0100,0.0027
8,252.0017,1477.0172,3.0812,1.3476,2.1524,4.9828,1.3476,0.0156,0.0108,0.0799
9,221.0125,1421.0029,3.1053,3.9403,6.0128,1.6741,1.0082,0.0470,0.0399,0.0010


候補画像は report.html で確認: C:\Users\mail\work\ikiikimake\outputs\makeup0923\report.html


## 目視で選んだ候補を固定

`APPROVED_RANK` を設定して実行します。既に同じ候補が `selected_pair.json` に固定済みなら、内容を確認してそのまま再利用します。別の候補へ勝手に上書きはしません。


In [10]:
import hashlib

if APPROVED_RANK is None:
    print('APPROVED_RANK=None: 候補固定はスキップしました。')
else:
    if isinstance(APPROVED_RANK, bool) or not isinstance(APPROVED_RANK, int):
        raise TypeError('APPROVED_RANK は整数で指定してください。')
    if not (1 <= APPROVED_RANK <= len(pairs)):
        raise ValueError(f'APPROVED_RANK は 1..{len(pairs)} の範囲です。')

    def sha256_file(path: Path) -> str:
        if not path.is_file():
            raise FileNotFoundError(path)
        digest = hashlib.sha256()
        with path.open('rb') as handle:
            for block in iter(lambda: handle.read(1024 * 1024), b''):
                digest.update(block)
        return digest.hexdigest()

    records = {record['frame_id']: record for record in scan_manifest['records']}
    pair = pairs[APPROVED_RANK - 1]
    selected = {
        'video_path': str(VIDEO.resolve()),
        'video_sha256': video_info['sha256'],
        'rank': APPROVED_RANK,
        'score': pair['score'],
        'before': {},
        'after': {},
    }
    for phase, key in (('before', 'before_id'), ('after', 'after_id')):
        record = records.get(pair[key])
        if record is None:
            raise RuntimeError(f'{phase} フレームが scan_manifest.json にありません: {pair[key]}')
        image_path = Path(record['image_path'])
        roi_dir = Path(record['roi_dir'])
        masks_path = roi_dir / 'roi_masks.npz'
        points_path = roi_dir / 'roi_points.json'
        overlay_path = roi_dir / 'roi_overlay.png'
        for required in (image_path, masks_path, points_path, overlay_path):
            if not required.is_file():
                raise FileNotFoundError(required)
        selected[phase] = {
            'frame_id': record['frame_id'],
            'timestamp_seconds': record['timestamp_seconds'],
            'image_path': str(image_path),
            'image_sha256': sha256_file(image_path),
            'roi_dir': str(roi_dir),
            'roi_masks_sha256': sha256_file(masks_path),
            'roi_points_sha256': sha256_file(points_path),
            'roi_overlay_sha256': sha256_file(overlay_path),
        }

    selected_path = OUTPUT / 'selected_pair.json'
    if selected_path.exists():
        existing = json.loads(selected_path.read_text(encoding='utf-8'))
        if existing != selected:
            raise FileExistsError(f'別内容の selected_pair.json が既にあります: {selected_path}')
        print('同じ候補が既に固定済みです:', selected_path)
    else:
        selected_path.write_text(json.dumps(selected, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
        print('固定しました:', selected_path)

FileExistsError: 別内容の selected_pair.json が既にあります: outputs\makeup0923\selected_pair.json

## 次

Lab解析は `video_selected_pair_lab.ipynb` を開き、同じ `VIDEO` ファイル名を指定して Run All するだけです。
